In [4]:
%pip install tabulate


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [64]:
import numpy as np
import numpy.ma as ma
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from recsysNN_utils import *
pd.set_option("display.precision", 1)

In [65]:
top10_df = pd.read_csv("./data/content_top10_df.csv")
bygenre_df = pd.read_csv("./data/content_bygenre_df.csv")
top10_df

,movie id,num ratings,ave rating,title,genres
0,4993,198,4.1,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
1,5952,188,4.0,"Lord of the Rings: The Two Towers, The",Adventure|Fantasy
2,7153,185,4.1,"Lord of the Rings: The Return of the King, The",Action|Adventure|Drama|Fantasy
3,4306,170,3.9,Shrek,Adventure|Animation|Children|Comedy|Fantasy|Ro...
4,58559,149,4.2,"Dark Knight, The",Action|Crime|Drama
5,6539,149,3.8,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy
6,79132,143,4.1,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller
7,6377,141,4.0,Finding Nemo,Adventure|Animation|Children|Comedy
8,4886,132,3.9,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy
9,7361,131,4.2,Eternal Sunshine of the Spotless Mind,Drama|Romance|Sci-Fi


In [66]:
bygenre_df

,genre,num movies,ave rating/genre,ratings per genre
0,Action,321,3.4,10377
1,Adventure,234,3.4,8785
2,Animation,76,3.6,2588
3,Children,69,3.4,2472
4,Comedy,326,3.4,8911
5,Crime,139,3.5,4671
6,Documentary,13,3.8,280
7,Drama,342,3.6,10201
8,Fantasy,124,3.4,4468
9,Horror,56,3.2,1345


In [67]:
pprint_train(user_train,user_features, uvs, u_s, maxcount =5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9


In [68]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,1,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,1,0,1,0,0,0,0,0,1
46970,2006,3.2,1,0,0,0,1,0,0,0,0,0,0,0,0,0
48516,2006,4.3,0,0,0,0,0,1,0,1,0,0,0,0,0,1
58559,2008,4.2,1,0,0,0,0,1,0,1,0,0,0,0,0,0


In [69]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  3.5 4.  4.  4.5]


In [70]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9


In [74]:
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

#INDEX
u_s = 3
i_s = 1

# Slice features
user_features_final = user_train[:, u_s:]  #  exactly 14 columns
item_features_final = item_train[:, i_s:]  # exactly 16 columns

#standard normalizers column-by-column
scalerUser = StandardScaler().fit(user_features_final)
user_train_scaled = scalerUser.transform(user_features_final)

scalerItem = StandardScaler().fit(item_features_final)
item_train_scaled = scalerItem.transform(item_features_final)

scalerTarget = MinMaxScaler((-1, 1)).fit(y_train.reshape(-1, 1))
y_train_scaled = scalerTarget.transform(y_train.reshape(-1, 1))

# Split datasets into an 80/20 division
user_train_split, user_test = train_test_split(user_train_scaled, train_size=0.80, shuffle=True, random_state=1)
item_train_split, item_test = train_test_split(item_train_scaled, train_size=0.80, shuffle=True, random_state=1)
y_train_split, y_test = train_test_split(y_train_scaled, train_size=0.80, shuffle=True, random_state=1)

print(f" User Feature Width: {user_train_split.shape[1]} ")
print(f" Movie Feature Width: {item_train_split.shape[1]} ")




 User Feature Width: 14 
 Movie Feature Width: 16 


In [75]:
tf.keras.backend.clear_session()
num_outputs = 32
tf.random.set_seed(1)

user_NN = tf.keras.models.Sequential([
    tf.keras.layers.Dense(units=256, activation='relu'),
    tf.keras.layers.Dense(units=128, activation='relu'),
    tf.keras.layers.Dense(units=num_outputs)
])

item_NN = tf.keras.models.Sequential([
    tf.keras.layers.Dense(units=256, activation='relu'),
    tf.keras.layers.Dense(units=128, activation='relu'),
    tf.keras.layers.Dense(units=num_outputs)
])

# Create inputs
input_user = tf.keras.layers.Input(shape=(14,))
vu = user_NN(input_user)
vu = tf.keras.utils.normalize(vu, axis=1)

input_item = tf.keras.layers.Input(shape=(16,))
vm = item_NN(input_item)
vm = tf.keras.utils.normalize(vm, axis=1)

output = tf.keras.layers.Dot(axes=1)([vu, vm])
model = tf.keras.Model(inputs=[input_user, input_item], outputs=output)

cost_fn = tf.keras.losses.MeanSquaredError()
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt, loss=cost_fn)


In [76]:
tf.random.set_seed(1)

model.fit(
    x=[user_train_split, item_train_split], 
    y=y_train_split, 
    epochs=30,
    batch_size=256
)



Epoch 1/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 1s 845us/step - loss: 0.1287
Epoch 2/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 831us/step - loss: 0.1170
Epoch 3/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 834us/step - loss: 0.1111
Epoch 4/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 828us/step - loss: 0.1074
Epoch 5/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 815us/step - loss: 0.1042
Epoch 6/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1011
Epoch 7/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 839us/step - loss: 0.0984
Epoch 8/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 824us/step - loss: 0.0964
Epoch 9/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 826us/step - loss: 0.0948
Epoch 10/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 824us/step - loss: 0.0932
Epoch 11/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 821us/step - loss: 0.0917
Epoch 12/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 819us/step - loss: 0.0902
Epoch 13/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 822us/step - loss: 0.0889
Epoch 14/30
160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 818us/step - loss: 0.0878
Epoch 15/30
160/160 ━━━━━━━━━━━

In [79]:
model.evaluate([user_test, item_test], y_test)

319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 247us/step - loss: 0.0864


0.08639614284038544

In [80]:
#Predictions

In [81]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

In [84]:
# PREDICTION SCALING FIX
user_vecs = gen_user_vecs(user_vec, len(item_vecs))

# Slice out the metadata columns FIRST before scaling
suser_vecs = scalerUser.transform(user_vecs[:, u_s:])  # Slices to 14 features
sitem_vecs = scalerItem.transform(item_vecs[:, i_s:])  # Slices to 16 features

# Make a prediction directly with your perfectly scaled features
y_p = model.predict([suser_vecs, sitem_vecs])

# Unscale y prediction and handle sorting as normal
y_pu = scalerTarget.inverse_transform(y_p)


# sort the results, highest prediction first
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display

print_pred_movies(sorted_ypu, sorted_items, movie_dict, maxcount = 10)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


y_p,movie id,rating ave,title,genres
4.1,137857,3.6,The Jungle Book (2016),Adventure|Drama|Fantasy
4.1,40815,3.8,Harry Potter and the Goblet of Fire (2005),Adventure|Fantasy|Thriller
4.1,8368,3.9,Harry Potter and the Prisoner of Azkaban (2004),Adventure|Fantasy
4.1,59501,3.5,"Chronicles of Narnia: Prince Caspian, The (2008)",Adventure|Children|Fantasy
4.1,54001,3.9,Harry Potter and the Order of the Phoenix (2007),Adventure|Drama|Fantasy
4.1,98809,3.8,"Hobbit: An Unexpected Journey, The (2012)",Adventure|Fantasy
4,4896,3.8,Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001),Adventure|Children|Fantasy
4,59387,4,"Fall, The (2006)",Adventure|Drama|Fantasy
4,81834,4,Harry Potter and the Deathly Hallows: Part 1 (2010),Action|Adventure|Fantasy
4,5816,3.6,Harry Potter and the Chamber of Secrets (2002),Adventure|Fantasy


In [ ]:
#Finding Similar Items

In [86]:
def sq_dist(a,b):
    d = np.sum((a-b)**2)
    return d

In [87]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3):0.3f}")

squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000


In [89]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features,))
vm_m = item_NN(input_item_m)
vm_m = tf.keras.utils.normalize(vm_m, axis=1)

model_m = tf.keras.Model(input_item_m, vm_m)
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 32)             │        41,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalize_2 (Normalize)         │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

In [97]:
scaled_item_vecs = scalerItem.transform(item_vecs[:, i_s:])
vms = model_m.predict(scaled_item_vecs)                  # Passes perfectly scaled data

print(f"size of all predicted movie feature vectors: {vms.shape}")


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 624us/step
size of all predicted movie feature vectors: (847, 32)


In [98]:
count = 50  # number of movies to display
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genres'],
                  movie_dict[movie2_id]['title'], movie_dict[movie1_id]['genres']]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow")
table


movie1,genres,movie2,genres
Save the Last Dance (2001),Drama|Romance,Mona Lisa Smile (2003),Drama|Romance
"Wedding Planner, The (2001)",Comedy|Romance,"Sweetest Thing, The (2002)",Comedy|Romance
Hannibal (2001),Horror|Thriller,Final Destination 2 (2003),Horror|Thriller
Saving Silverman (Evil Woman) (2001),Comedy|Romance,"Wedding Planner, The (2001)",Comedy|Romance
Down to Earth (2001),Comedy|Fantasy|Romance,Bewitched (2005),Comedy|Fantasy|Romance
"Mexican, The (2001)",Action|Comedy,Rush Hour 2 (2001),Action|Comedy
15 Minutes (2001),Thriller,Panic Room (2002),Thriller
Enemy at the Gates (2001),Drama,"Aviator, The (2004)",Drama
Heartbreakers (2001),Comedy|Crime|Romance,Fun with Dick and Jane (2005),Comedy|Crime|Romance
Spy Kids (2001),Action|Adventure|Children|Comedy,Scooby-Doo (2002),Action|Adventure|Children|Comedy
